# Project Finalization and Packaging

## Stage 15: Final Report, Documentation, and ZIP Package

This notebook validates the completed project, extracts final results,
generates project documentation, creates defense notes, builds a project
manifest, and packages the required files into the final submission ZIP.

The final package excludes development-only directories such as virtual
environments, Git metadata, IDE settings, and notebook checkpoints.

In [2]:
import hashlib
import importlib.metadata
import json
import re
import shutil
import sys
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display


STUDENT_ID = "YOUR_STUDENT_ID"
FIRST_NAME = "YOUR_FIRST_NAME"
LAST_NAME = "YOUR_LAST_NAME"


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"
MODELS_DIR = OUTPUTS_DIR / "models"
REPORT_DIR = PROJECT_ROOT / "report"
PRESENTATION_DIR = PROJECT_ROOT / "presentation"
DIST_DIR = PROJECT_ROOT / "dist"


REPORT_DIR.mkdir(parents=True, exist_ok=True)
PRESENTATION_DIR.mkdir(parents=True, exist_ok=True)
DIST_DIR.mkdir(parents=True, exist_ok=True)


placeholder_values = {
    "1401012261003",
    "Amirhossein",
    "Khadivi",
}


provided_identity_values = {
    STUDENT_ID,
    FIRST_NAME,
    LAST_NAME,
}


if placeholder_values.intersection(
    provided_identity_values
):
    raise ValueError(
        "Replace the student identity placeholders before continuing."
    )


def sanitize_name(value):
    cleaned_value = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(value).strip(),
    )

    return cleaned_value.strip("_")


safe_student_id = sanitize_name(STUDENT_ID)
safe_first_name = sanitize_name(FIRST_NAME)
safe_last_name = sanitize_name(LAST_NAME)


PACKAGE_NAME = (
    f"DMProject_{safe_student_id}_"
    f"{safe_first_name}_{safe_last_name}"
)


PACKAGE_DIR = DIST_DIR / PACKAGE_NAME
ZIP_PATH = DIST_DIR / f"{PACKAGE_NAME}.zip"


print("Project root:")
print(PROJECT_ROOT)

print("\nPackage name:")
print(PACKAGE_NAME)

print("\nZIP path:")
print(ZIP_PATH)

Project root:
E:\Projects\DataMining

Package name:
DMProject_YOUR_STUDENT_ID_YOUR_FIRST_NAME_YOUR_LAST_NAME

ZIP path:
E:\Projects\DataMining\dist\DMProject_YOUR_STUDENT_ID_YOUR_FIRST_NAME_YOUR_LAST_NAME.zip


In [3]:
REQUIRED_PROJECT_FILES = [
    DATA_DIR / "raw" / "data.csv",
    NOTEBOOKS_DIR
    / "01_data_inspection_and_preprocessing.ipynb",
    NOTEBOOKS_DIR
    / "02_exploratory_data_analysis.ipynb",
    NOTEBOOKS_DIR
    / "03_modeling_and_evaluation.ipynb",
    NOTEBOOKS_DIR
    / "04_python_vs_pandas_performance.ipynb",
    NOTEBOOKS_DIR
    / "05_finalization_and_packaging.ipynb",
    TABLES_DIR
    / "01_target_distribution.csv",
    TABLES_DIR
    / "03_missing_values_report.csv",
    TABLES_DIR
    / "04_adjusted_iqr_outlier_report.csv",
    TABLES_DIR
    / "05_distribution_statistics.csv",
    TABLES_DIR
    / "06_target_distribution.csv",
    TABLES_DIR
    / "07_numeric_target_correlation.csv",
    TABLES_DIR
    / "08_eda_key_findings.csv",
    TABLES_DIR
    / "09_split_assignments.csv",
    TABLES_DIR
    / "10_svm_test_results.csv",
    TABLES_DIR
    / "11_knn_test_results.csv",
    TABLES_DIR
    / "12_decision_tree_test_results.csv",
    TABLES_DIR
    / "13_final_model_comparison.csv",
    TABLES_DIR
    / "13_recommended_model_summary.csv",
    TABLES_DIR
    / "14_performance_summary.csv",
    TABLES_DIR
    / "14_speedup_summary.csv",
]


file_validation_rows = []


for file_path in REQUIRED_PROJECT_FILES:
    file_validation_rows.append(
        {
            "relative_path": str(
                file_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "exists": file_path.exists(),
            "size_bytes": (
                file_path.stat().st_size
                if file_path.exists()
                else 0
            ),
        }
    )


file_validation_report = pd.DataFrame(
    file_validation_rows
)


display(file_validation_report)


missing_required_files = (
    file_validation_report.loc[
        ~file_validation_report["exists"],
        "relative_path",
    ].tolist()
)


if missing_required_files:
    raise FileNotFoundError(
        "Required project files are missing:\n"
        + "\n".join(missing_required_files)
    )


print("All required project files are available.")

,relative_path,exists,size_bytes
0,data\raw\data.csv,True,79346
1,notebooks\01_data_inspection_and_preprocessing...,True,350328
2,notebooks\02_exploratory_data_analysis.ipynb,True,1628488
3,notebooks\03_modeling_and_evaluation.ipynb,True,948875
4,notebooks\04_python_vs_pandas_performance.ipynb,True,103622
5,notebooks\05_finalization_and_packaging.ipynb,True,4580
6,outputs\tables\01_target_distribution.csv,True,93
7,outputs\tables\03_missing_values_report.csv,True,383
8,outputs\tables\04_adjusted_iqr_outlier_report.csv,True,472
9,outputs\tables\05_distribution_statistics.csv,True,888


All required project files are available.


In [4]:
final_model_comparison = pd.read_csv(
    TABLES_DIR
    / "13_final_model_comparison.csv"
)


recommended_model_summary = pd.read_csv(
    TABLES_DIR
    / "13_recommended_model_summary.csv"
)


speedup_summary = pd.read_csv(
    TABLES_DIR
    / "14_speedup_summary.csv"
)


eda_key_findings = pd.read_csv(
    TABLES_DIR
    / "08_eda_key_findings.csv"
)


target_distribution = pd.read_csv(
    TABLES_DIR
    / "01_target_distribution.csv"
)


missing_values_report = pd.read_csv(
    TABLES_DIR
    / "03_missing_values_report.csv"
)


recommended_model = (
    recommended_model_summary.iloc[0]
)


benchmark_result = speedup_summary.iloc[0]


print("Recommended model:")
display(recommended_model_summary.round(4))


print("Final model comparison:")
display(final_model_comparison.round(4))


print("Benchmark result:")
display(speedup_summary.round(4))

Recommended model:


,selection_rank,model_name,algorithm,model_detail,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,training_seconds,prediction_seconds,class_4_recall
0,1,knn_k5_with_dataset,KNN,k=5 | dataset=with_dataset,0.5924,0.3997,0.5665,0.3997,0.4172,0.5782,0.0614,0.1222,0.1667


Final model comparison:


,model_name,algorithm,model_detail,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,training_seconds,prediction_seconds,class_4_recall
0,knn_k5_with_dataset,KNN,k=5 | dataset=with_dataset,0.5924,0.3997,0.5665,0.3997,0.4172,0.5782,0.0614,0.1222,0.1667
1,svm_rbf_with_dataset_balanced,RBF SVM,kernel=rbf | dataset=with_dataset | class_weig...,0.5380,0.4054,0.3998,0.4054,0.3918,0.5657,0.0829,0.0782,0.1667
2,optimized_svm_median_standard,Optimized SVM,imputation=median | scaler=standard | dataset=...,0.5054,0.4453,0.3910,0.4453,0.3818,0.5364,0.1006,0.0203,0.6667
3,svm_linear_with_dataset_balanced,Linear SVM,kernel=linear | dataset=with_dataset | class_w...,0.5054,0.4453,0.3910,0.4453,0.3818,0.5364,0.0948,0.0237,0.6667
4,decision_tree_depth_5_with_dataset_none,Decision Tree,depth=5 | dataset=with_dataset | class_weight=...,0.5598,0.3432,0.2964,0.3432,0.3155,0.5286,0.0543,0.0415,0.0000


Benchmark result:


,raw_python_average_seconds,pandas_numpy_average_seconds,raw_python_to_pandas_speedup,faster_method,benchmark_rows,benchmark_repeats
0,0.3383,0.0359,9.4112,Pandas and NumPy,92000,5


In [5]:
def dataframe_to_markdown(
    input_frame,
    maximum_rows=None,
    decimal_places=4,
):
    frame = input_frame.copy()

    if maximum_rows is not None:
        frame = frame.head(maximum_rows)

    for column_name in frame.columns:
        if pd.api.types.is_float_dtype(
            frame[column_name]
        ):
            frame[column_name] = (
                frame[column_name]
                .round(decimal_places)
            )

    headers = [
        str(column_name)
        for column_name in frame.columns
    ]

    separator = [
        "---"
        for _ in headers
    ]

    markdown_rows = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(separator) + " |",
    ]

    for _, row in frame.iterrows():
        row_values = []

        for value in row:
            if pd.isna(value):
                formatted_value = ""
            else:
                formatted_value = str(value)

            formatted_value = formatted_value.replace(
                "|",
                "\\|",
            )

            row_values.append(formatted_value)

        markdown_rows.append(
            "| "
            + " | ".join(row_values)
            + " |"
        )

    return "\n".join(markdown_rows)

In [9]:
recommended_algorithm = str(
    recommended_model.get(
        "algorithm",
        "Not available",
    )
)


recommended_macro_f1 = float(
    recommended_model.get(
        "macro_f1",
        0.0,
    )
)


recommended_accuracy = float(
    recommended_model.get(
        "accuracy",
        0.0,
    )
)


recommended_balanced_accuracy = float(
    recommended_model.get(
        "balanced_accuracy",
        0.0,
    )
)


recommended_class_4_recall = float(
    recommended_model.get(
        "class_4_recall",
        0.0,
    )
)


benchmark_speedup = float(
    benchmark_result.get(
        "raw_python_to_pandas_speedup",
        0.0,
    )
)


benchmark_faster_method = str(
    benchmark_result.get(
        "faster_method",
        "Not available",
    )
)


readme_content = f"""# Multiclass Heart Disease Classification

## Student Information

- Student ID: {STUDENT_ID}
- First Name: {FIRST_NAME}
- Last Name: {LAST_NAME}

## Project Overview

This project applies data preprocessing, exploratory data analysis, and
multiclass classification techniques to the Heart Disease dataset.

The target variable contains five classes from 0 to 4.

## Dataset

- Number of records: 920
- Number of original columns: 16
- Identifier column: id
- Target column: num
- Target classes: 0, 1, 2, 3, 4

## Project Structure

```text
data/
notebooks/
outputs/
report/
presentation/
README.md
requirements.txt
"""

In [10]:

model_comparison_markdown = (
    dataframe_to_markdown(
        final_model_comparison[
            [
                "algorithm",
                "accuracy",
                "balanced_accuracy",
                "macro_precision",
                "macro_recall",
                "macro_f1",
                "weighted_f1",
                "class_4_recall",
            ]
        ],
        decimal_places=4,
    )
)


target_distribution_markdown = (
    dataframe_to_markdown(
        target_distribution,
        decimal_places=4,
    )
)


missing_summary_markdown = (
    dataframe_to_markdown(
        missing_values_report,
        maximum_rows=16,
        decimal_places=4,
    )
)


eda_findings_markdown = (
    dataframe_to_markdown(
        eda_key_findings,
        maximum_rows=30,
        decimal_places=4,
    )
)


report_content = f"""# Final Data Mining Project Report

## Student Information

- Student ID: {STUDENT_ID}
- First Name: {FIRST_NAME}
- Last Name: {LAST_NAME}

## 1. Project Objective

The objective of this project was to preprocess, analyze, and classify
multiclass Heart Disease records. The target variable represents disease
severity using classes 0 through 4.

## 2. Dataset Description

The dataset contains 920 records and 16 original columns. The `id`
column is an identifier and was excluded from machine-learning models.
The `num` column is the multiclass target.

### Target Distribution

{target_distribution_markdown}

## 3. Data Quality and Preprocessing

The dataset contains missing values in several numeric and categorical
features.

### Missing-Value Summary

{missing_summary_markdown}

The preprocessing workflow included:

1. Replacing suspicious zero values in `trestbps` and `chol` with missing
   values.
2. Comparing mean and median imputation.
3. Representing missing categorical values as an explicit category.
4. Applying One-Hot Encoding to categorical features.
5. Comparing StandardScaler and MinMaxScaler.
6. analyzing outliers using the IQR method.
7. performing all learned preprocessing operations only on training data.

## 4. Exploratory Data Analysis

The exploratory analysis included histograms, KDE plots, skewness,
kurtosis, boxplots, categorical frequency charts, correlation matrices,
feature-target relationships, class imbalance analysis, and dataset
source analysis.

### Main EDA Findings

{eda_findings_markdown}

No valid temporal column was available. Therefore, temporal trend
analysis was documented as not applicable.

## 5. Data Splitting Strategy

The dataset was divided into 80 percent training data and 20 percent
test data.

Stratification was used to preserve the multiclass target distribution.
The random state was fixed at 42.

## 6. Machine-Learning Models

The following classifiers were evaluated:

- Linear Support Vector Machine
- RBF Support Vector Machine
- K-Nearest Neighbors with multiple neighborhood sizes
- Decision Tree with multiple depth and class-weight configurations

Model configurations were selected using five-fold stratified
cross-validation on the training data.

The test set was used only for final evaluation.

## 7. Evaluation Metrics

The following metrics were calculated:

- Accuracy
- Balanced Accuracy
- Macro Precision
- Macro Recall
- Macro F1
- Weighted F1
- Confusion Matrix
- Per-class Recall

Macro F1 was selected as the primary metric because the target classes
were imbalanced.

## 8. Final Model Comparison

{model_comparison_markdown}

## 9. Recommended Model

- Algorithm: {recommended_algorithm}
- Accuracy: {recommended_accuracy:.4f}
- Balanced Accuracy: {recommended_balanced_accuracy:.4f}
- Macro F1: {recommended_macro_f1:.4f}
- Class 4 Recall: {recommended_class_4_recall:.4f}

The recommended model was selected primarily by Macro F1, followed by
class 4 recall and balanced accuracy.

## 10. Bonus Performance Comparison

The same preprocessing and descriptive-statistics operation was
implemented using raw Python loops and using vectorized Pandas/NumPy
operations.

- Faster method: {benchmark_faster_method}
- Raw Python to Pandas/NumPy ratio: {benchmark_speedup:.4f}

Vectorized operations were generally faster because they reduce Python
interpreter overhead and use optimized compiled routines.

## 11. Limitations

- The target classes are strongly imbalanced.
- Class 4 contains relatively few observations.
- Missingness differs across dataset collection centers.
- The `dataset` feature may capture source-specific patterns.
- The dataset contains no valid temporal feature.
- Results should not be interpreted as clinical medical advice.

## 12. Conclusion

The project completed the required preprocessing, exploratory analysis,
classification, model comparison, and performance benchmark.

Leakage-safe pipelines were used throughout the modeling workflow.
The final recommendation balances overall performance with performance
on minority classes.
"""


FINAL_REPORT_PATH = (
    REPORT_DIR / "final_report.md"
)


FINAL_REPORT_PATH.write_text(
    report_content,
    encoding="utf-8",
)


print("Final report created successfully.")
print(FINAL_REPORT_PATH)

Final report created successfully.
E:\Projects\DataMining\report\final_report.md


In [11]:
defense_notes_content = f"""# Defense Notes

## Project Summary

This project performs multiclass classification on the Heart Disease
dataset. The target contains five classes from 0 to 4.

## Important Numbers

- Dataset rows: 920
- Original columns: 16
- Training rows: 736
- Test rows: 184
- Train-test ratio: 80/20
- Random state: 42
- Cross-validation folds: 5
- Recommended algorithm: {recommended_algorithm}
- Recommended Macro F1: {recommended_macro_f1:.4f}

## Common Defense Questions

### 1. Why was the ID column removed?

The ID column identifies records and does not represent a medical
feature. Including it could create meaningless patterns.

### 2. Why was stratification used?

The target classes are imbalanced. Stratification preserves similar
class proportions in the training and test sets.

### 3. Why is accuracy not sufficient?

A model can obtain acceptable accuracy by focusing on the majority
class. Macro F1 gives equal importance to every target class.

### 4. Why was Macro F1 the primary metric?

The dataset contains five imbalanced classes. Macro F1 evaluates each
class equally before calculating the average.

### 5. What is data leakage?

Data leakage occurs when information from validation or test data
influences model training. All imputers, encoders, and scalers were
fitted only on training data.

### 6. Why was a Pipeline used?

The Pipeline combines preprocessing and classification. It ensures that
the same transformations are applied consistently and prevents leakage
during cross-validation.

### 7. Why were suspicious zeros replaced?

Zero blood pressure and zero cholesterol were treated as unavailable
measurements based on a rule-based data-quality assumption.

### 8. What is the difference between Linear and RBF SVM?

Linear SVM creates linear decision boundaries. RBF SVM can model
nonlinear class relationships.

### 9. Why does KNN require scaling?

KNN uses distance calculations. Features with larger numeric scales can
dominate the distance without scaling.

### 10. Why does Decision Tree not require scaling?

Decision Trees split features using thresholds. Their decisions are not
based on geometric distance.

### 11. What does class weight balanced do?

It assigns greater importance to minority classes and lower importance
to majority classes based on their frequencies.

### 12. Why was the dataset feature tested both ways?

The dataset column identifies collection centers. It may improve
performance but can also cause the model to learn source-specific
patterns.

### 13. Why was temporal analysis not performed?

The dataset contains no valid date or time feature. The dataset source
column is not a temporal variable.

### 14. What does a Confusion Matrix show?

It shows the number of correct and incorrect predictions for every
actual and predicted class combination.

### 15. Why was the test set evaluated only after model selection?

Repeatedly selecting models based on test performance would make the
test set part of the training decision process.

## Final Recommendation

The selected model is {recommended_algorithm} with a test Macro F1 of
{recommended_macro_f1:.4f}, balanced accuracy of
{recommended_balanced_accuracy:.4f}, and class 4 recall of
{recommended_class_4_recall:.4f}.
"""


DEFENSE_NOTES_PATH = (
    PRESENTATION_DIR / "defense_notes.md"
)


DEFENSE_NOTES_PATH.write_text(
    defense_notes_content,
    encoding="utf-8",
)


print("Defense notes created successfully.")
print(DEFENSE_NOTES_PATH)

Defense notes created successfully.
E:\Projects\DataMining\presentation\defense_notes.md


In [12]:
REQUIRED_PACKAGES = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scipy",
    "scikit-learn",
    "joblib",
    "jupyter",
    "ipykernel",
]


requirement_rows = []
requirement_lines = []


for package_name in REQUIRED_PACKAGES:
    try:
        package_version = (
            importlib.metadata.version(
                package_name
            )
        )

        requirement_lines.append(
            f"{package_name}=={package_version}"
        )

        requirement_rows.append(
            {
                "package": package_name,
                "version": package_version,
                "available": True,
            }
        )

    except importlib.metadata.PackageNotFoundError:
        requirement_rows.append(
            {
                "package": package_name,
                "version": "",
                "available": False,
            }
        )


requirements_content = (
    "\n".join(requirement_lines)
    + "\n"
)


REQUIREMENTS_PATH = (
    PROJECT_ROOT / "requirements.txt"
)


REQUIREMENTS_PATH.write_text(
    requirements_content,
    encoding="utf-8",
)


requirements_report = pd.DataFrame(
    requirement_rows
)


display(requirements_report)


print("Requirements file created successfully.")
print(REQUIREMENTS_PATH)

,package,version,available
0,numpy,2.2.6,True
1,pandas,2.3.3,True
2,matplotlib,3.10.9,True
3,seaborn,0.13.2,True
4,scipy,1.15.3,True
5,scikit-learn,1.7.2,True
6,joblib,1.5.3,True
7,jupyter,1.1.1,True
8,ipykernel,7.3.0,True


Requirements file created successfully.
E:\Projects\DataMining\requirements.txt


In [16]:
project_summary = {
    "student": {
        "student_id": 1401012261003,
        "first_name": "Amirhossein",
        "last_name": "Khadivi",
    },
    "dataset": {
        "rows": 920,
        "original_columns": 16,
        "target": "num",
        "target_classes": [
            0,
            1,
            2,
            3,
            4,
        ],
    },
    "modeling": {
        "train_rows": 736,
        "test_rows": 184,
        "test_size": 0.20,
        "random_state": 42,
        "cross_validation_folds": 5,
    },
    "recommended_model": {
        "algorithm": recommended_algorithm,
        "accuracy": recommended_accuracy,
        "balanced_accuracy": (
            recommended_balanced_accuracy
        ),
        "macro_f1": recommended_macro_f1,
        "class_4_recall": (
            recommended_class_4_recall
        ),
    },
    "bonus_benchmark": {
        "faster_method": (
            benchmark_faster_method
        ),
        "speedup_ratio": benchmark_speedup,
    },
    "generated_at": datetime.now().isoformat(
        timespec="seconds"
    ),
}


PROJECT_SUMMARY_PATH = (
    REPORT_DIR / "project_summary.json"
)


PROJECT_SUMMARY_PATH.write_text(
    json.dumps(
        project_summary,
        indent=2,
        ensure_ascii=True,
    ),
    encoding="utf-8",
)


print("Project summary created successfully.")
print(PROJECT_SUMMARY_PATH)

Project summary created successfully.
E:\Projects\DataMining\report\project_summary.json


In [17]:
EXCLUDED_DIRECTORY_NAMES = {
    ".git",
    ".idea",
    ".venv",
    "venv",
    "__pycache__",
    ".ipynb_checkpoints",
    "dist",
}


def should_include_file(file_path):
    relative_parts = file_path.relative_to(
        PROJECT_ROOT
    ).parts

    return not any(
        part in EXCLUDED_DIRECTORY_NAMES
        for part in relative_parts
    )


def calculate_sha256(file_path):
    hash_object = hashlib.sha256()

    with file_path.open("rb") as input_file:
        while True:
            data_chunk = input_file.read(
                1024 * 1024
            )

            if not data_chunk:
                break

            hash_object.update(data_chunk)

    return hash_object.hexdigest()


manifest_rows = []


for file_path in sorted(
    PROJECT_ROOT.rglob("*")
):
    if not file_path.is_file():
        continue

    if not should_include_file(file_path):
        continue

    manifest_rows.append(
        {
            "relative_path": str(
                file_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "size_bytes": (
                file_path.stat().st_size
            ),
            "sha256": calculate_sha256(
                file_path
            ),
        }
    )


project_manifest = pd.DataFrame(
    manifest_rows
)


MANIFEST_PATH = (
    TABLES_DIR
    / "15_project_manifest.csv"
)


project_manifest.to_csv(
    MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)


print(f"Manifest files: {len(project_manifest)}")
display(project_manifest.head(20))

Manifest files: 149


,relative_path,size_bytes,sha256
0,.gitignore,461,e3f875f386278bacfd80170faab892ca5521f172c88fff...
1,data\raw\data.csv,79346,574f2fa2b43012fa25fca4fcdb36bd7c6bccdd0af4242f...
2,notebooks\01_data_inspection_and_preprocessing...,350328,bc399ee19d3593b41df77aeb776b5320d77e54d3f33b0a...
3,notebooks\02_exploratory_data_analysis.ipynb,1628488,18146e1d4339195ad3e952f2f8e91f7fdb3146b6d640a4...
4,notebooks\03_modeling_and_evaluation.ipynb,948875,0cd1adea64fc48a359a21f42fffe62c611f6ea5ccfa01e...
5,notebooks\04_python_vs_pandas_performance.ipynb,103622,54c93fdd77475573e37423d1254b1ca1546ace09361c81...
6,notebooks\05_finalization_and_packaging.ipynb,67483,010bcdeaf6141e06d218293879b8bc964832095593ceb2...
7,outputs\figures\03_missing_by_dataset_heatmap.png,169922,2c9263479f70340a7f56e0a2c07dd8c07c7a7f95d8c3d1...
8,outputs\figures\03_missing_values_bar.png,90170,a72fb5ebe10038aec77e2c60e05ad156aef9717f226ae2...
9,outputs\figures\04_boxplot_age.png,29622,541525db25deb86fd63730f6e775b3992957fe6f7ce1fb...


In [19]:
README_PATH = PROJECT_ROOT / "README.md"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"


print("README path:")
print(README_PATH)
print(README_PATH.exists())


print("\nRequirements path:")
print(REQUIREMENTS_PATH)
print(REQUIREMENTS_PATH.exists())

README path:
E:\Projects\DataMining\README.md
True

Requirements path:
E:\Projects\DataMining\requirements.txt
True
